# Mnemo: Stage 1 (video → action log) + Stage 2 (questions) on a T4 GPU

**Stage 1:** a vision-language model (Qwen3-VL-4B) watches every 4 seconds of video and writes what the
camera wearer did: *any* action, *any* object (e.g. "closes | fridge door | kitchen"). Only this text is stored.

**Stage 2:** questions like *"did I close the fridge?"* are answered from that log only, with times.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Upload the latest `mnemo_code.zip` to your
Drive folder first (on the laptop: `python -m tools.make_colab_zip`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/Final Year Project'   # folder with mnemo_code.zip
assert os.path.exists(f'{WORK}/mnemo_code.zip'), f'Upload mnemo_code.zip to {WORK} first'
!rm -rf /content/mnemo && mkdir -p /content/mnemo && cd /content/mnemo && unzip -q "{WORK}/mnemo_code.zip"
%cd /content/mnemo
!ls

## 1. Install + load the model (first time: ~9 GB download, a few minutes)

In [ ]:
!pip install -q -U transformers accelerate
import torch, transformers, time
print('transformers', transformers.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (switch to T4!)')
import config as C
from modules.vlm_logger import VLMDescriber, ActionLogger
t = time.time()
describer = VLMDescriber(C.VLM_MODEL)
print(f'{C.VLM_MODEL} loaded in {time.time()-t:.0f}s | GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 2. Choose a video
Default: an EPIC-KITCHENS clip from your Drive by a person the models never trained on (P26). You can also
upload your own video to the Drive folder and put its path here.

In [ ]:
VIDEO = f'{WORK}/epic_clips/P26_110_w178.mp4'
assert os.path.exists(VIDEO), VIDEO

# What really happens in this EPIC clip (all actions, from the official annotations), for comparison
import csv, urllib.request
ann = '/content/EPIC_100_train.csv'
if not os.path.exists(ann):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-100-annotations/master/EPIC_100_train.csv', ann)
name = os.path.splitext(os.path.basename(VIDEO))[0]
vid, _, t0 = name.rpartition('_w')
def secs(s):
    h, m, x = s.split(':'); return int(h)*3600 + int(m)*60 + float(x)
truth = sorted((secs(r['start_timestamp']) - float(t0), r['narration']) for r in csv.DictReader(open(ann))
               if r['video_id'] == vid and 0 <= secs(r['start_timestamp']) - float(t0) < 60)
print(f'TRUE actions in {name} ({len(truth)}):')
for t, n in truth:
    print(f'  {t:5.1f}s  {n}')

## 3. Stage 1: video → action log

In [ ]:
from datetime import datetime
from modules.storage import EventStore
from modules.vlm_logger import ActionLogger
store = EventStore(f'{WORK}/mnemo_log.db')
store.clear_activities()                                  # one demo video per run
START = datetime.now().replace(hour=9, minute=0, second=0, microsecond=0)   # pretend it was filmed at 9:00
rows, stats = ActionLogger(describer, store).run(VIDEO, start_time=START)
print(stats)
print(f"\nreal-time factor {stats['real_time_factor']}: below 1.0 means the model keeps up with live video")

## 4. The stored log (this is ALL that is kept: text, no video)

In [ ]:
from query.llm_qa import fmt_row
for r in store.activities():
    print(fmt_row(r))

## 5. Stage 2: ask questions
Change the questions to things you can see in the TRUE list above. Answers use only the log.

In [ ]:
from datetime import timedelta
from query.llm_qa import answer
NOW = START + timedelta(hours=1)
QUESTIONS = [                                 # for the P26 clip; the last one is NOT in the video
    'What did I do in the kitchen this morning?',
    'Did I open the oven?',
    'Did I close the oven?',
    'Where did I put the plate?',
    'Did I take my medicine?',
]
for q in QUESTIONS:
    r = answer(q, store, llm=describer, now=NOW)
    print(f'Q: {q}\nA: {r["answer"]}\n   (used {len(r["lines"])} log lines)\n')

## 6. (Optional) speed comparison with the smaller 2B model
If the real-time factor above is well above 1.0, try the 2B model: faster, usually somewhat less accurate.

In [ ]:
del describer; import gc; gc.collect(); torch.cuda.empty_cache()
describer = VLMDescriber('Qwen/Qwen3-VL-2B-Instruct')
store2 = EventStore('/content/log_2b.db')
rows2, stats2 = ActionLogger(describer, store2).run(VIDEO, start_time=START)
print(stats2)
for r in store2.activities():
    print(fmt_row(r))

## 7. Use the log on the laptop
`My Drive/Final Year Project/mnemo_log.db` holds the log. Download it to the laptop and run:
```bash
python mnemo.py --db mnemo_log.db activities
python mnemo.py --db mnemo_log.db qa "did I open the fridge?"
```
(Without a GPU the laptop lists the matching log rows instead of writing a sentence.)